# 模型可視化與解釋性 (Model Visualization and Interpretability)
:label:`sec_model_visualization`

深度學習模型常被稱為"黑盒"，但我們可以通過多種可視化技術來理解模型的決策過程。

## 為什麼需要模型解釋性？

1. **調試模型**：發現模型學到了什麼
2. **建立信任**：在醫療、自動駕駛等關鍵領域尤為重要
3. **改進模型**：找出模型的弱點
4. **滿足法規**：GDPR等要求模型可解釋

## 本節內容

1. **特徵可視化**：CNN學到了什麼
2. **激活圖可視化**：哪些區域被激活
3. **GradCAM**：模型關注哪裡
4. **濾波器可視化**：卷積核學到的模式
5. **誤分類分析**：模型在哪裡出錯

In [ ]:
# 安裝必要套件
# !pip install pytorch-grad-cam opencv-python captum

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2

# GradCAM
from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. 特徵圖可視化

查看CNN中間層的激活圖，了解模型如何處理圖像。

In [ ]:
def visualize_feature_maps(model, image, layer_names):
    """
    可視化指定層的特徵圖
    
    Args:
        model: 模型
        image: 輸入圖像 tensor
        layer_names: 要可視化的層名稱列表
    """
    model.eval()
    
    # 註冊 hook 來捕獲中間層輸出
    activations = {}
    
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook
    
    # 註冊hooks
    handles = []
    for name, layer in model.named_modules():
        if name in layer_names:
            handles.append(layer.register_forward_hook(get_activation(name)))
    
    # 前向傳播
    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
    
    # 移除hooks
    for handle in handles:
        handle.remove()
    
    # 可視化
    for layer_name in layer_names:
        if layer_name in activations:
            feature_maps = activations[layer_name].cpu().squeeze()
            
            # 選擇前16個通道
            n_features = min(16, feature_maps.shape[0])
            
            fig, axes = plt.subplots(4, 4, figsize=(12, 12))
            fig.suptitle(f'Feature Maps - {layer_name}', fontsize=16, fontweight='bold')
            
            for i in range(n_features):
                ax = axes[i // 4, i % 4]
                ax.imshow(feature_maps[i], cmap='viridis')
                ax.axis('off')
                ax.set_title(f'Channel {i}')
            
            plt.tight_layout()
            plt.show()

# 示例使用
model = models.resnet18(pretrained=True).to(device)

# 準備測試圖像
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 如果有測試圖像，可以這樣使用：
# image = Image.open('test.jpg')
# image_tensor = transform(image)
# visualize_feature_maps(model, image_tensor, ['layer1', 'layer2', 'layer3'])

## 2. GradCAM：類激活映射

**Gradient-weighted Class Activation Mapping (GradCAM)** 可以顯示模型在做預測時關注圖像的哪些區域。

In [ ]:
def apply_gradcam(model, image_path, target_layers, target_category=None):
    """
    應用 GradCAM 並可視化
    
    Args:
        model: CNN模型
        image_path: 圖像路徑
        target_layers: 目標層（通常是最後一個卷積層）
        target_category: 目標類別（None表示預測類別）
    """
    # 讀取並預處理圖像
    rgb_img = np.array(Image.open(image_path).convert('RGB').resize((224, 224))) / 255.0
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    input_tensor = transform(rgb_img).unsqueeze(0)
    
    # 創建 GradCAM 對象
    cam = GradCAM(model=model, target_layers=target_layers)
    
    # 如果未指定目標類別，使用模型預測的類別
    targets = None
    if target_category is not None:
        targets = [ClassifierOutputTarget(target_category)]
    
    # 生成CAM
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0, :]
    
    # 疊加到原圖
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    
    # 顯示結果
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(rgb_img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(grayscale_cam, cmap='jet')
    axes[1].set_title('GradCAM Heatmap')
    axes[1].axis('off')
    
    axes[2].imshow(visualization)
    axes[2].set_title('GradCAM Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return grayscale_cam, visualization

# 示例：對 ResNet18 使用 GradCAM
# model = models.resnet18(pretrained=True).eval()
# target_layers = [model.layer4[-1]]
# cam, vis = apply_gradcam(model, 'test.jpg', target_layers)

## 3. 濾波器可視化

可視化卷積層學到的濾波器/卷積核。

In [ ]:
def visualize_conv_filters(model, layer_name, num_filters=64):
    """
    可視化卷積層的濾波器
    
    Args:
        model: 模型
        layer_name: 層名稱
        num_filters: 顯示的濾波器數量
    """
    # 獲取指定層的權重
    for name, module in model.named_modules():
        if name == layer_name and isinstance(module, nn.Conv2d):
            weights = module.weight.data.cpu()
            
            # 標準化到 [0, 1]
            weights = weights - weights.min()
            weights = weights / weights.max()
            
            # 選擇要顯示的濾波器數量
            num_filters = min(num_filters, weights.shape[0])
            
            # 計算網格大小
            grid_size = int(np.ceil(np.sqrt(num_filters)))
            
            fig, axes = plt.subplots(grid_size, grid_size, figsize=(15, 15))
            fig.suptitle(f'Convolutional Filters - {layer_name}', fontsize=16, fontweight='bold')
            
            for i in range(grid_size * grid_size):
                ax = axes[i // grid_size, i % grid_size]
                
                if i < num_filters:
                    # 獲取單個濾波器
                    filt = weights[i]
                    
                    # 如果是彩色濾波器(3通道)，直接顯示
                    if filt.shape[0] == 3:
                        filt = filt.permute(1, 2, 0)
                        ax.imshow(filt)
                    else:
                        # 否則顯示第一個通道
                        ax.imshow(filt[0], cmap='gray')
                    
                    ax.set_title(f'Filter {i}', fontsize=8)
                
                ax.axis('off')
            
            plt.tight_layout()
            plt.show()
            break

# 示例
model = models.resnet18(pretrained=True)
visualize_conv_filters(model, 'conv1', num_filters=64)

## 4. 使用 Captum 進行高級解釋

[Captum](https://captum.ai/) 是 PyTorch 的模型可解釋性庫。

In [ ]:
from captum.attr import IntegratedGradients, Saliency, DeepLift
from captum.attr import visualization as viz

def interpret_with_captum(model, image_tensor, method='integrated_gradients'):
    """
    使用 Captum 進行模型解釋
    
    Args:
        model: 模型
        image_tensor: 輸入圖像張量
        method: 'integrated_gradients', 'saliency', 'deeplift'
    """
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    
    # 獲取預測
    output = model(image_tensor)
    pred_class = output.argmax(dim=1).item()
    
    # 選擇歸因方法
    if method == 'integrated_gradients':
        ig = IntegratedGradients(model)
        attributions = ig.attribute(image_tensor, target=pred_class)
    elif method == 'saliency':
        saliency = Saliency(model)
        attributions = saliency.attribute(image_tensor, target=pred_class)
    elif method == 'deeplift':
        dl = DeepLift(model)
        attributions = dl.attribute(image_tensor, target=pred_class)
    
    # 可視化
    attributions = attributions.squeeze().cpu().detach().numpy()
    attributions = np.transpose(attributions, (1, 2, 0))
    
    original_image = np.transpose(image_tensor.squeeze().cpu().numpy(), (1, 2, 0))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(original_image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(np.abs(attributions).sum(axis=2), cmap='hot')
    axes[1].set_title(f'{method.replace("_", " ").title()}\nPredicted Class: {pred_class}')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return attributions

# 示例
# model = models.resnet18(pretrained=True).to(device)
# attr = interpret_with_captum(model, image_tensor, method='integrated_gradients')

## 5. 誤分類分析

分析模型在哪些樣本上出錯，幫助改進模型。

In [ ]:
def analyze_misclassifications(model, test_loader, class_names, num_samples=16):
    """
    分析並可視化誤分類樣本
    
    Args:
        model: 模型
        test_loader: 測試數據加載器
        class_names: 類別名稱列表
        num_samples: 顯示的樣本數量
    """
    model.eval()
    misclassified = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            
            # 找出誤分類的樣本
            mask = predicted != labels
            for i in range(len(mask)):
                if mask[i]:
                    misclassified.append({
                        'image': images[i].cpu(),
                        'true': labels[i].item(),
                        'pred': predicted[i].item(),
                        'confidence': torch.softmax(outputs[i], 0).max().item()
                    })
                    
                    if len(misclassified) >= num_samples:
                        break
            
            if len(misclassified) >= num_samples:
                break
    
    # 可視化
    grid_size = int(np.ceil(np.sqrt(num_samples)))
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(15, 15))
    fig.suptitle('Misclassified Samples', fontsize=16, fontweight='bold')
    
    for i in range(grid_size * grid_size):
        ax = axes[i // grid_size, i % grid_size]
        
        if i < len(misclassified):
            sample = misclassified[i]
            
            # 反標準化並轉換為可顯示格式
            img = sample['image'].permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min())
            
            ax.imshow(img)
            ax.set_title(
                f"True: {class_names[sample['true']]}\n"
                f"Pred: {class_names[sample['pred']]}\n"
                f"Conf: {sample['confidence']:.2f}",
                fontsize=9
            )
        
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# 示例
# analyze_misclassifications(model, test_loader, class_names=['cat', 'dog'])

## 6. 小結

### 可視化技術對比

| 技術 | 用途 | 優點 | 缺點 |
|------|------|------|------|
| **特徵圖** | 理解中間層表示 | 直觀 | 高層抽象難理解 |
| **GradCAM** | 定位關鍵區域 | 類別特定 | 分辨率低 |
| **Integrated Gradients** | 像素級重要性 | 理論基礎好 | 計算慢 |
| **濾波器可視化** | 理解卷積核 | 適合淺層 | 深層難解釋 |

### 實踐建議

1. **從GradCAM開始**：最直觀易用
2. **結合多種方法**：互相驗證
3. **關注誤分類**：發現模型弱點
4. **迭代改進**：基於可視化結果調整模型

### 下一步

- [實用技巧](11_practical_tips.ipynb)：模型訓練和調優
- [完整項目](12_complete_project.ipynb)：端到端實戰

## 參考資源

- [GradCAM論文](https://arxiv.org/abs/1610.02391)
- [pytorch-grad-cam](https://github.com/jacobgil/pytorch-grad-cam)
- [Captum文檔](https://captum.ai/)
- [Distill.pub - Feature Visualization](https://distill.pub/2017/feature-visualization/)